# 01 - Athena Setup

Sets up Glue database and tables so Athena can query the shared Yelp dataset.

**Shared S3 bucket:** `s3://aai-540-group1-yelp-reviews/`

**Regular AWS users only (one-time IAM setup):**
1. IAM → Roles → `AmazonSageMaker-ExecutionRole-...` → Permissions → attach `AWSGlueServiceRole`
2. Trust relationships → Edit → add `glue.amazonaws.com` to the Service list

**AWS Academy users:** no setup needed, LabRole already has required permissions

## 0. Install Dependencies

In [1]:
import importlib, subprocess

def install_if_missing(package):
    if importlib.util.find_spec(package) is None:
        print(f'Installing {package}...')
        subprocess.run(['pip', 'install', package, '--quiet'], check=True)
        print(f'{package} installed')
    else:
        print(f'{package} already installed, skipping')

install_if_missing('pyarrow')
install_if_missing('fastparquet')

pyarrow already installed, skipping
fastparquet already installed, skipping


## 1. Configuration

Only change `YOUR_NAME` to your initials — everything else is shared.

In [2]:
# Shared config — do not change
REGION        = 'us-east-2'
SOURCE_BUCKET = 'aai-540-group1-yelp-reviews'
GLUE_DB       = 'yelp_reviews_db'

# Per-person config — change to your initials to avoid Athena result collisions
YOUR_NAME     = 'qmou'   # <-- CHANGE THIS
ATHENA_RESULTS = f's3://{SOURCE_BUCKET}/athena-results/{YOUR_NAME}/'

print(f'Region         : {REGION}')
print(f'Source bucket  : s3://{SOURCE_BUCKET}/')
print(f'Glue database  : {GLUE_DB}')
print(f'Athena results : {ATHENA_RESULTS}')

Region         : us-east-2
Source bucket  : s3://aai-540-group1-yelp-reviews/
Glue database  : yelp_reviews_db
Athena results : s3://aai-540-group1-yelp-reviews/athena-results/qmou/


## 2. AWS Credentials

In [3]:
import boto3, json, time

session = boto3.Session(region_name=REGION)

def get_role():
    # Option 1: standard sagemaker
    try:
        import sagemaker
        return sagemaker.get_execution_role()
    except Exception:
        pass

    # Option 2: SageMaker Studio metadata (regular AWS)
    try:
        with open('/opt/ml/metadata/resource-metadata.json') as f:
            meta = json.load(f)
        return meta['ExecutionRoleArn']
    except Exception:
        pass

    # Option 3: classic notebook instance
    try:
        with open('/opt/ml/metadata/resource-name') as f:
            nb_name = f.read().strip()
        sm = session.client('sagemaker')
        return sm.describe_notebook_instance(NotebookInstanceName=nb_name)['RoleArn']
    except Exception:
        pass

    # Option 4: AWS Academy
    try:
        iam = session.client('iam')
        return iam.get_role(RoleName='LabRole')['Role']['Arn']
    except Exception:
        pass

    raise RuntimeError(
        'Could not detect IAM role automatically.\n'
        'Paste your role ARN into MANUAL_ROLE_ARN below.'
    )

MANUAL_ROLE_ARN = ''  # paste ARN here if auto-detection fails
role = MANUAL_ROLE_ARN if MANUAL_ROLE_ARN else get_role()

print(f'Role   : {role}')
print(f'Region : {session.region_name}')

Role   : arn:aws:iam::203012547555:role/service-role/AmazonSageMaker-ExecutionRole-20260521T133544
Region : us-east-2


## 3. Verify S3 Access

In [4]:
s3 = session.client('s3')

response = s3.list_objects_v2(Bucket=SOURCE_BUCKET, Prefix='data/')
print(f'Files in s3://{SOURCE_BUCKET}/data/:')
for obj in response.get('Contents', []):
    size_mb = obj['Size'] / (1024 ** 2)
    if size_mb > 0:
        print(f'  {obj["Key"]} ({size_mb:.1f} MB)')

Files in s3://aai-540-group1-yelp-reviews/data/:
  data/2019/yelp_reviews_2019.parquet (243.3 MB)
  data/2020_2022/yelp_reviews_2020-2022.parquet (315.0 MB)


## 4. Create Glue Database and Tables

Skips if database or tables already exist.

In [5]:
glue = session.client('glue')

# Create database
try:
    glue.create_database(DatabaseInput={'Name': GLUE_DB})
    print(f'Database {GLUE_DB} created')
except glue.exceptions.AlreadyExistsException:
    print(f'Database {GLUE_DB} already exists, skipping')

# Yelp dataset schema
yelp_columns = [
    {'Name': 'review_id',   'Type': 'string'},
    {'Name': 'user_id',     'Type': 'string'},
    {'Name': 'business_id', 'Type': 'string'},
    {'Name': 'stars',       'Type': 'bigint'},
    {'Name': 'useful',      'Type': 'bigint'},
    {'Name': 'funny',       'Type': 'bigint'},
    {'Name': 'cool',        'Type': 'bigint'},
    {'Name': 'text',        'Type': 'string'},
    {'Name': 'date',        'Type': 'timestamp'},
]

# Tables pointing to S3 folders (not files)
tables_to_create = {
    'yelp_reviews_2019':     f's3://{SOURCE_BUCKET}/data/2019/',
    'yelp_reviews_2020_2022': f's3://{SOURCE_BUCKET}/data/2020_2022/',
}

for table_name, s3_path in tables_to_create.items():
    try:
        glue.create_table(
            DatabaseName=GLUE_DB,
            TableInput={
                'Name': table_name,
                'StorageDescriptor': {
                    'Columns': yelp_columns,
                    'Location': s3_path,
                    'InputFormat': 'org.apache.hadoop.mapred.TextInputFormat',
                    'OutputFormat': 'org.apache.hadoop.hive.ql.io.HiveIgnoreKeyTextOutputFormat',
                    'SerdeInfo': {
                        'SerializationLibrary': 'org.apache.hadoop.hive.ql.io.parquet.serde.ParquetHiveSerDe',
                        'Parameters': {'serialization.format': '1'}
                    },
                },
                'TableType': 'EXTERNAL_TABLE',
                'Parameters': {'classification': 'parquet'}
            }
        )
        print(f'Table {table_name} created')
    except glue.exceptions.AlreadyExistsException:
        print(f'Table {table_name} already exists, skipping')

Database yelp_reviews_db already exists, skipping


Table yelp_reviews_2019 created


Table yelp_reviews_2020_2022 created


## 5. Verify Athena Queries

In [6]:
athena = session.client('athena')

def run_athena_query(sql):
    response = athena.start_query_execution(
        QueryString=sql,
        QueryExecutionContext={'Database': GLUE_DB},
        ResultConfiguration={'OutputLocation': ATHENA_RESULTS}
    )
    query_id = response['QueryExecutionId']
    while True:
        status = athena.get_query_execution(QueryExecutionId=query_id)
        state = status['QueryExecution']['Status']['State']
        if state == 'SUCCEEDED':
            break
        elif state in ['FAILED', 'CANCELLED']:
            reason = status['QueryExecution']['Status']['StateChangeReason']
            raise Exception(f'Query {state}: {reason}')
        time.sleep(2)
    return athena.get_query_results(QueryExecutionId=query_id)

for table in ['yelp_reviews_2019', 'yelp_reviews_2020_2022']:
    sql = f'SELECT COUNT(*) as row_count FROM {table}'
    results = run_athena_query(sql)
    count = results['ResultSet']['Rows'][1]['Data'][0]['VarCharValue']
    print(f'{table}: {count} rows')

yelp_reviews_2019: 907284 rows


yelp_reviews_2020_2022: 1204411 rows


## 6. Save Config for Other Notebooks

In [7]:
import json

config = {
    'REGION':         REGION,
    'SOURCE_BUCKET':  SOURCE_BUCKET,
    'GLUE_DB':        GLUE_DB,
    'ATHENA_RESULTS': ATHENA_RESULTS,
    'TABLES':         list(tables_to_create.keys())
}

with open('project_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('Config saved to project_config.json')
print(json.dumps(config, indent=2))

Config saved to project_config.json
{
  "REGION": "us-east-2",
  "SOURCE_BUCKET": "aai-540-group1-yelp-reviews",
  "GLUE_DB": "yelp_reviews_db",
  "ATHENA_RESULTS": "s3://aai-540-group1-yelp-reviews/athena-results/qmou/",
  "TABLES": [
    "yelp_reviews_2019",
    "yelp_reviews_2020_2022"
  ]
}
